# RAG vs. No-RAG — Multi-Family LLM Evaluation
## Notebook 02 — Vector DB: Corpus A (Župan dataset, English, nomic-embed-text)

**Corpus A** — Structured English Q&A-derived documents.
- 522 unique accounts, English descriptions
- Embedding: `nomic-embed-text` (English-optimised)
- Collection: `corpus_a_nomic`

Serves as the **structured baseline** RAG condition (replicates conference paper setup).


---
## ⚙️ Configuration

In [1]:
# ── CONFIGURATION — edit only this cell ──────────────────────────────────────

OLLAMA_BASE_URL = "http://100.119.36.113:11434"   # ← set your server IP

DATA_DIR    = "data"
CHROMA_DIR  = f"{DATA_DIR}/chroma_db"
RESULTS_DIR = f"{DATA_DIR}/results"
FIGURES_DIR = f"{DATA_DIR}/figures"

TOP_K       = 5
TEMPERATURE = 0
MAX_TOKENS  = 512

# ── Embedding models ───────────────────────────────────────────────────────────
EMBED_MODELS = {
    "nomic":   "nomic-embed-text",       # English-optimised
    "qwen3s":  "qwen3-embedding:0.6b",   # Multilingual lightweight
    "qwen3m":  "qwen3-embedding:4b",     # Multilingual mid-size
}

# ── ChromaDB collections ───────────────────────────────────────────────────────
COLLECTIONS = {
    "rag_a":        ("corpus_a_nomic",    "nomic"),
    "rag_b_nomic":  ("corpus_b_nomic",    "nomic"),
    "rag_b_qwen3s": ("corpus_b_qwen3s",   "qwen3s"),
    "rag_b_qwen3m": ("corpus_b_qwen3m",   "qwen3m"),
}
# Format: condition → (collection_name, embed_key)

CONDITIONS = ["no_rag", "rag_a", "rag_b_nomic", "rag_b_qwen3s", "rag_b_qwen3m"]

COND_LABELS = {
    "no_rag":        "No RAG",
    "rag_a":         "RAG-A (EN, nomic)",
    "rag_b_nomic":   "RAG-B (HR, nomic)",
    "rag_b_qwen3s":  "RAG-B (HR, qwen3-0.6b)",
    "rag_b_qwen3m":  "RAG-B (HR, qwen3-4b)",
}

# ── Model families ─────────────────────────────────────────────────────────────
MODELS = {
    "Qwen3.5": [
        "qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:4b", "qwen3.5:9b",
        "qwen3.5:27b",  "qwen3.5:35b", "qwen3.5:122b",
    ],
    "Llama": [
        "llama3.2:1b", "llama3.2:3b", "llama3.1:8b", "llama3.3:70b",
    ],
    "Gemma": [
        "gemma3:1b", "gemma3:4b", "gemma3:12b", "gemma3:27b",
    ],
    "Phi": [
        "phi4-mini", "phi4",
    ],
    "DeepSeek": [
        "deepseek-r1:1.5b", "deepseek-r1:7b", "deepseek-r1:14b",
        "deepseek-r1:32b",  "deepseek-r1:70b",
    ],
}
ALL_MODELS = [m for family in MODELS.values() for m in family]

MODEL_PARAMS = {
    "qwen3.5:0.8b":0.8,  "qwen3.5:2b":2,    "qwen3.5:4b":4,
    "qwen3.5:9b":9,      "qwen3.5:27b":27,   "qwen3.5:35b":35,
    "qwen3.5:122b":122,
    "llama3.2:1b":1,     "llama3.2:3b":3,    "llama3.1:8b":8,
    "llama3.3:70b":70,
    "gemma3:1b":1,       "gemma3:4b":4,      "gemma3:12b":12,
    "gemma3:27b":27,
    "phi4-mini":3.8,     "phi4":14,
    "deepseek-r1:1.5b":1.5, "deepseek-r1:7b":7,  "deepseek-r1:14b":14,
    "deepseek-r1:32b":32,   "deepseek-r1:70b":70,
}

import os
for d in [DATA_DIR, CHROMA_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

total_calls = len(ALL_MODELS) * len(CONDITIONS) * 1060
print("✓ Configuration loaded")
print(f"  Models:      {len(ALL_MODELS)}  ({', '.join(MODELS.keys())})")
print(f"  Conditions:  {len(CONDITIONS)}")
print(f"  Questions:   1,060")
print(f"  Total calls: {total_calls:,}")
print()
print("  Embedding models:")
for k, v in EMBED_MODELS.items():
    print(f"    {k:8s}: {v}")


✓ Configuration loaded
  Models:      22  (Qwen3.5, Llama, Gemma, Phi, DeepSeek)
  Conditions:  5
  Questions:   1,060
  Total calls: 116,600

  Embedding models:
    nomic   : nomic-embed-text
    qwen3s  : qwen3-embedding:0.6b
    qwen3m  : qwen3-embedding:4b


---
## 📦 Dependencies

In [2]:
import subprocess
subprocess.run(
    ["pip", "install", "-q", "--root-user-action=ignore", "chromadb", "requests", "tqdm"],
    capture_output=True
)
print("✓ OK")


✓ OK


---
## 🔌 Check Ollama

In [3]:
import requests, time, json

def check_ollama(url):
    try:
        r = requests.get(f"{url}/api/tags", timeout=5)
        r.raise_for_status()
        return True, [m["name"] for m in r.json().get("models", [])]
    except Exception as e:
        return False, str(e)

ok, available = check_ollama(OLLAMA_BASE_URL)
if not ok:
    raise RuntimeError(f"Ollama unavailable: {available}")

model = EMBED_MODELS["nomic"]
status = "✓" if any(model in m for m in available) else "✗ MISSING"
print(f"✓ Ollama OK")
print(f"  {status}  {model}")
if status.startswith("✗"):
    print(f"     Run: docker exec -it ollama ollama pull {model}")


✓ Ollama OK
  ✓  nomic-embed-text


---
## 🔧 Build Corpus A documents

In [4]:
import re

with open("accounts.jsonl") as f:
    raw_data = [json.loads(l) for l in f if l.strip()]

def parse_example(example):
    convs = example["conversations"]
    u = next(c for c in convs if c["role"] == "user")
    a = next(c for c in convs if c["role"] == "assistant")
    try:
        answer = json.loads(a["content"])
    except:
        answer = {}
    return {"answer": answer}

all_parsed = [parse_example(ex) for ex in raw_data]

docs_by_account = {}
for ex in all_parsed:
    a  = ex["answer"]
    if not isinstance(a, dict): continue
    num  = a.get("account_number","").strip()
    name = a.get("english_name", a.get("name","")).strip()
    hrv  = a.get("croatian_name","").strip()
    if not num: continue
    if num not in docs_by_account:
        parts = [f"Account number: {num}"]
        if name: parts.append(f"English name: {name}")
        if hrv:  parts.append(f"Croatian name: {hrv}")
        docs_by_account[num] = {
            "id":             f"account_{num}",
            "account_number": num,
            "text":           "\n".join(parts),
            "metadata":       {"account_number": num,
                               "english_name":   name,
                               "croatian_name":  hrv},
        }

corpus_a_docs = list(docs_by_account.values())
print(f"✓ Corpus A: {len(corpus_a_docs)} documents")
print()
print("Example:")
print("-" * 55)
print(corpus_a_docs[0]["text"])
print("-" * 55)


✓ Corpus A: 522 documents

Example:
-------------------------------------------------------
Account number: 000
English name: Receivables for recoded but not paid in capital in joint-stock company (analytics by Potraživanja za upisani a neuplaćeni dionički kapital (analitika po upisnicima)
-------------------------------------------------------


---
## 🗄️ Build ChromaDB collection

In [5]:
import chromadb
from tqdm import tqdm

COLLECTION_NAME = "corpus_a_nomic"
EMBED_MODEL     = EMBED_MODELS["nomic"]

def get_embedding(text, model=EMBED_MODEL):
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=30
    )
    r.raise_for_status()
    return r.json()["embedding"]

# Test
vec = get_embedding("Account number: 020\nEnglish name: Land")
print(f"✓ Embedding dim: {len(vec)}")

client   = chromadb.PersistentClient(path=CHROMA_DIR)
existing = [c.name for c in client.list_collections()]

if COLLECTION_NAME in existing:
    coll = client.get_collection(COLLECTION_NAME)
    if coll.count() == len(corpus_a_docs):
        print(f"✓ '{COLLECTION_NAME}' already complete ({coll.count()} docs)")
    else:
        client.delete_collection(COLLECTION_NAME)
        coll = client.create_collection(COLLECTION_NAME,
                                        metadata={"hnsw:space": "cosine"})
        print(f"  Recreating...")
else:
    coll = client.create_collection(COLLECTION_NAME,
                                    metadata={"hnsw:space": "cosine"})
    print(f"✓ Created '{COLLECTION_NAME}'")


✓ Embedding dim: 768
✓ Created 'corpus_a_nomic'


In [6]:
BATCH_SIZE = 50
errors     = []

if coll.count() < len(corpus_a_docs):
    t0 = time.time()
    for i in tqdm(range(0, len(corpus_a_docs), BATCH_SIZE), ncols=70,
                  desc="Embedding Corpus A"):
        batch = corpus_a_docs[i:i+BATCH_SIZE]
        ids, embeddings, documents, metadatas = [], [], [], []
        for doc in batch:
            try:
                ids.append(doc["id"])
                embeddings.append(get_embedding(doc["text"]))
                documents.append(doc["text"])
                metadatas.append(doc["metadata"])
            except Exception as e:
                errors.append({"id": doc["id"], "error": str(e)})
        if ids:
            coll.add(ids=ids, embeddings=embeddings,
                     documents=documents, metadatas=metadatas)
    print(f"\n✓ Done in {time.time()-t0:.0f}s  Errors: {len(errors)}")

print(f"  Documents: {coll.count()}")


Embedding Corpus A: 100%|█████████████| 11/11 [00:18<00:00,  1.66s/it]


✓ Done in 18s  Errors: 0
  Documents: 522


---
## 🔍 Test retrieval

In [7]:
def retrieve_a(query, k=TOP_K):
    vec = get_embedding(query)
    res = coll.query(query_embeddings=[vec], n_results=k,
                     include=["metadatas","distances"])
    return [{"account_number": m.get("account_number",""),
             "english_name":   m.get("english_name",""),
             "similarity":     round(1-d, 4)}
            for m, d in zip(res["metadatas"][0], res["distances"][0])]

for query in [
    "What is the account code for Land?",
    "What is the account description for 020?",
    "What is the account code for Cash in bank?",
]:
    res = retrieve_a(query)
    print(f"QUERY: '{query}'")
    for i, r in enumerate(res, 1):
        print(f"  [{i}] {r['account_number']:>5}  sim={r['similarity']:.3f}  "
              f"{r['english_name'][:45]}")
    print()

print("✓ Corpus A ready")
print("Next: 03_build_vectordb_corpus_b.ipynb")


QUERY: 'What is the account code for Land?'
  [1]   103  sim=0.682  Foreign currency accounts Devizni računi
  [2]   016  sim=0.663  Cryptocurencies Kriptovalute
  [3]   020  sim=0.657  Land Zemljišta
  [4]   486  sim=0.651  Gifts/donations up to
  [5]   500  sim=0.650  Operation I Pogon I

QUERY: 'What is the account description for 020?'
  [1]   020  sim=0.797  Land Zemljišta
  [2]   030  sim=0.675  Plant Postrojenja
  [3]   024  sim=0.665  Flats for employees Stanovi za vlastite zapos
  [4]   103  sim=0.658  Foreign currency accounts Devizni računi
  [5]   190  sim=0.647  Prepaid expenses and accrued income Unaprijed

QUERY: 'What is the account code for Cash in bank?'
  [1]   100  sim=0.739  Cash in bank (transaction accounts) Transakci
  [2]   103  sim=0.702  Foreign currency accounts Devizni računi
  [3]   444  sim=0.651  Value adjustment of deposit on banks, bills o
  [4]   016  sim=0.648  Cryptocurencies Kriptovalute
  [5]   104  sim=0.645  Letter of credit issued in foreign ba

In [8]:
import requests
r = requests.get("http://100.119.36.113:11434/api/tags")
models = [m["name"] for m in r.json()["models"]]
for m in sorted(models):
    if "embed" in m or "nomic" in m:
        print(m)

mxbai-embed-large:latest
nomic-embed-text:latest
qwen3-embedding:0.6b
qwen3-embedding:4b
